# Renewable Energy Shortfall Analysis for Germany (2012–2020)

This notebook combines:

- **Germany electricity data** from Open Power System Data (OPSD)
- **Weather data** from NASA POWER for five German cities

The goal is to create a unified dataset for analyzing and forecasting renewable energy shortfalls.

## Variables Used

### Energy Variables
- **load_mw**: Germany electricity load in MW
- **wind_generation_mw**: Germany wind generation in MW
- **solar_generation_mw**: Germany solar generation in MW
- **shortfall_mw**: Difference between load and renewable generation

### Weather Variables
- **irradiance_wm2**: Average solar irradiance across selected German cities
- **wind_speed_10m_ms**: Average wind speed at 10 m
- **temperature_2m_c**: Average air temperature at 2 m

## Study Period
2012 to 2020

## Selected Locations for Weather Averaging
- Berlin
- Hamburg
- Munich
- Frankfurt
- Cologne

In [12]:
# Imports
import pandas as pd
import requests

## 1. Download Germany Energy Data from OPSD

This section loads hourly electricity load, wind generation, and solar generation data for Germany from the Open Power System Data platform.

In [13]:
opsd_url = "https://data.open-power-system-data.org/time_series/latest/time_series_60min_singleindex.csv"

opsd = pd.read_csv(
    opsd_url,
    usecols=[
        "utc_timestamp",
        "DE_load_actual_entsoe_transparency",
        "DE_wind_generation_actual",
        "DE_solar_generation_actual",
    ]
)

opsd["utc_timestamp"] = pd.to_datetime(opsd["utc_timestamp"], utc=True)

opsd = opsd[
    (opsd["utc_timestamp"] >= "2012-01-01") &
    (opsd["utc_timestamp"] < "2021-01-01")
].copy()

opsd = opsd.rename(columns={
    "DE_load_actual_entsoe_transparency": "load_mw",
    "DE_wind_generation_actual": "wind_generation_mw",
    "DE_solar_generation_actual": "solar_generation_mw"
})

opsd = opsd.dropna(subset=["load_mw", "wind_generation_mw", "solar_generation_mw"])

opsd["shortfall_mw"] = (
    opsd["load_mw"]
    - opsd["wind_generation_mw"]
    - opsd["solar_generation_mw"]
)

print("OPSD shape:", opsd.shape)
opsd.head()

OPSD shape: (50295, 5)


,utc_timestamp,load_mw,solar_generation_mw,wind_generation_mw,shortfall_mw
8,2015-01-01 07:00:00+00:00,41133.0,71.0,10208.0,30854.0
9,2015-01-01 08:00:00+00:00,42963.0,773.0,10029.0,32161.0
10,2015-01-01 09:00:00+00:00,45088.0,2117.0,10550.0,32421.0
11,2015-01-01 10:00:00+00:00,47013.0,3364.0,11390.0,32259.0
12,2015-01-01 11:00:00+00:00,48159.0,4198.0,12103.0,31858.0


## 2. Define German Cities for NASA POWER Weather Data

Since NASA POWER provides point-based weather data, several cities are selected across Germany to create a more representative national average.

In [14]:
cities = {
    "Berlin":    {"lat": 52.52, "lon": 13.405},
    "Hamburg":   {"lat": 53.5511, "lon": 9.9937},
    "Munich":    {"lat": 48.1351, "lon": 11.5820},
    "Frankfurt": {"lat": 50.1109, "lon": 8.6821},
    "Cologne":   {"lat": 50.9375, "lon": 6.9603},
}

cities

{'Berlin': {'lat': 52.52, 'lon': 13.405},
 'Hamburg': {'lat': 53.5511, 'lon': 9.9937},
 'Munich': {'lat': 48.1351, 'lon': 11.582},
 'Frankfurt': {'lat': 50.1109, 'lon': 8.6821},
 'Cologne': {'lat': 50.9375, 'lon': 6.9603}}

## 3. Create a Function to Download NASA POWER Hourly Weather Data

The function below requests hourly weather data for a given latitude and longitude in UTC format.

In [15]:
def get_nasa_power_hourly(lat, lon, start="20120101", end="20201231"):
    url = (
        "https://power.larc.nasa.gov/api/temporal/hourly/point"
        "?parameters=ALLSKY_SFC_SW_DWN,WS10M,T2M"
        "&community=RE"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start={start}"
        f"&end={end}"
        "&format=JSON"
        "&time-standard=UTC"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data = response.json()

    params = data["properties"]["parameter"]

    df = pd.DataFrame({
        "utc_timestamp": list(params["ALLSKY_SFC_SW_DWN"].keys()),
        "irradiance_wm2": list(params["ALLSKY_SFC_SW_DWN"].values()),
        "wind_speed_10m_ms": list(params["WS10M"].values()),
        "temperature_2m_c": list(params["T2M"].values()),
    })

    df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"], format="%Y%m%d%H", utc=True)

    return df

## 4. Download NASA POWER Data for All Selected Cities

This step downloads hourly solar irradiance, wind speed, and temperature for each selected city.

In [16]:
weather_frames = []

for city, coords in cities.items():
    print(f"Downloading NASA POWER data for {city}...")
    city_df = get_nasa_power_hourly(coords["lat"], coords["lon"])
    city_df = city_df.rename(columns={
        "irradiance_wm2": f"{city}_irradiance_wm2",
        "wind_speed_10m_ms": f"{city}_wind_speed_10m_ms",
        "temperature_2m_c": f"{city}_temperature_2m_c",
    })
    weather_frames.append(city_df)

## 5. Merge Weather Data Across Cities

Each city's weather dataframe is merged using the UTC timestamp.

In [17]:
weather = weather_frames[0]

for df_city in weather_frames[1:]:
    weather = weather.merge(df_city, on="utc_timestamp", how="inner")

print("Weather shape:", weather.shape)
weather.head()

Weather shape: (78912, 16)


,utc_timestamp,Berlin_irradiance_wm2,Berlin_wind_speed_10m_ms,Berlin_temperature_2m_c,Hamburg_irradiance_wm2,Hamburg_wind_speed_10m_ms,Hamburg_temperature_2m_c,Munich_irradiance_wm2,Munich_wind_speed_10m_ms,Munich_temperature_2m_c,Frankfurt_irradiance_wm2,Frankfurt_wind_speed_10m_ms,Frankfurt_temperature_2m_c,Cologne_irradiance_wm2,Cologne_wind_speed_10m_ms,Cologne_temperature_2m_c
0,2012-01-01 00:00:00+00:00,0.0,3.24,0.25,0.0,2.86,3.79,0.0,4.03,1.21,0.0,4.20,6.12,0.0,6.01,9.12
1,2012-01-01 01:00:00+00:00,0.0,2.84,0.32,0.0,3.74,4.41,0.0,4.13,1.41,0.0,4.07,6.05,0.0,5.78,9.24
2,2012-01-01 02:00:00+00:00,0.0,2.43,0.36,0.0,4.75,5.16,0.0,4.47,1.94,0.0,4.18,6.20,0.0,5.97,9.31
3,2012-01-01 03:00:00+00:00,0.0,2.37,0.41,0.0,5.41,5.93,0.0,4.64,2.43,0.0,4.18,6.45,0.0,6.42,9.52
4,2012-01-01 04:00:00+00:00,0.0,2.65,0.88,0.0,5.72,6.65,0.0,4.80,2.73,0.0,4.26,6.65,0.0,6.87,9.77


## 6. Average the Weather Variables Across Germany

The city-based weather variables are averaged to create one representative weather series for Germany.

In [18]:
irradiance_cols = [f"{city}_irradiance_wm2" for city in cities]
wind_cols = [f"{city}_wind_speed_10m_ms" for city in cities]
temp_cols = [f"{city}_temperature_2m_c" for city in cities]

weather["irradiance_wm2"] = weather[irradiance_cols].mean(axis=1)
weather["wind_speed_10m_ms"] = weather[wind_cols].mean(axis=1)
weather["temperature_2m_c"] = weather[temp_cols].mean(axis=1)

weather_avg = weather[[
    "utc_timestamp",
    "irradiance_wm2",
    "wind_speed_10m_ms",
    "temperature_2m_c"
]].copy()

print("Averaged weather shape:", weather_avg.shape)
weather_avg.head()

Averaged weather shape: (78912, 4)


,utc_timestamp,irradiance_wm2,wind_speed_10m_ms,temperature_2m_c
0,2012-01-01 00:00:00+00:00,0.0,4.068,4.098
1,2012-01-01 01:00:00+00:00,0.0,4.112,4.286
2,2012-01-01 02:00:00+00:00,0.0,4.360,4.594
3,2012-01-01 03:00:00+00:00,0.0,4.604,4.948
4,2012-01-01 04:00:00+00:00,0.0,4.860,5.336


## 7. Merge Energy and Weather Data

This step combines the OPSD Germany energy data with the averaged NASA POWER weather data.

In [19]:
merged = opsd.merge(weather_avg, on="utc_timestamp", how="inner")

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (50295, 8)


,utc_timestamp,load_mw,solar_generation_mw,wind_generation_mw,shortfall_mw,irradiance_wm2,wind_speed_10m_ms,temperature_2m_c
0,2015-01-01 07:00:00+00:00,41133.0,71.0,10208.0,30854.0,3.652,3.858,-2.142
1,2015-01-01 08:00:00+00:00,42963.0,773.0,10029.0,32161.0,48.378,4.148,-1.322
2,2015-01-01 09:00:00+00:00,45088.0,2117.0,10550.0,32421.0,105.020,4.456,-0.006
3,2015-01-01 10:00:00+00:00,47013.0,3364.0,11390.0,32259.0,138.192,4.400,0.858
4,2015-01-01 11:00:00+00:00,48159.0,4198.0,12103.0,31858.0,154.970,4.156,1.624


## 8. Add Metadata and Units

This table summarizes the meaning and units of the variables in the final dataset.

In [20]:
units = {
    "utc_timestamp": "datetime (UTC)",
    "load_mw": "MW",
    "wind_generation_mw": "MW",
    "solar_generation_mw": "MW",
    "shortfall_mw": "MW",
    "irradiance_wm2": "W/m^2",
    "wind_speed_10m_ms": "m/s",
    "temperature_2m_c": "degC",
}

metadata_table = pd.DataFrame({
    "column": list(units.keys()),
    "unit": list(units.values()),
    "description": [
        "Timestamp",
        "Germany electricity load",
        "Germany wind generation",
        "Germany solar generation",
        "Load minus wind and solar generation",
        "Average solar irradiance across five German cities",
        "Average wind speed at 10 m across five German cities",
        "Average air temperature at 2 m across five German cities",
    ]
})

metadata_table

,column,unit,description
0,utc_timestamp,datetime (UTC),Timestamp
1,load_mw,MW,Germany electricity load
2,wind_generation_mw,MW,Germany wind generation
3,solar_generation_mw,MW,Germany solar generation
4,shortfall_mw,MW,Load minus wind and solar generation
5,irradiance_wm2,W/m^2,Average solar irradiance across five German ci...
6,wind_speed_10m_ms,m/s,Average wind speed at 10 m across five German ...
7,temperature_2m_c,degC,Average air temperature at 2 m across five Ger...


## 9. Dataset Overview

This section checks the time range, missing values, and summary statistics of the merged dataset.

In [21]:
print("Time range:")
print("Start:", merged["utc_timestamp"].min())
print("End:  ", merged["utc_timestamp"].max())

print("\nMissing values:")
print(merged.isna().sum())

print("\nSummary statistics:")
merged.describe()

Time range:
Start: 2015-01-01 07:00:00+00:00
End:   2020-09-30 23:00:00+00:00

Missing values:
utc_timestamp          0
load_mw                0
solar_generation_mw    0
wind_generation_mw     0
shortfall_mw           0
irradiance_wm2         0
wind_speed_10m_ms      0
temperature_2m_c       0
dtype: int64

Summary statistics:


,load_mw,solar_generation_mw,wind_generation_mw,shortfall_mw,irradiance_wm2,wind_speed_10m_ms,temperature_2m_c
count,50295.000000,50295.000000,50295.000000,50295.000000,50295.000000,50295.000000,50295.000000
mean,55487.982980,4566.170673,11553.928363,39367.883945,131.760759,4.002748,10.379830
std,10016.027177,6940.373607,9078.275500,12105.356056,194.429932,1.725024,8.193967
min,31307.000000,0.000000,135.000000,648.000000,0.000000,0.846000,-12.560000
25%,47100.000000,0.000000,4505.500000,31493.000000,0.000000,2.758000,3.706000
50%,55090.000000,172.000000,9016.000000,39726.000000,7.204000,3.612000,9.818000
75%,64305.000000,7342.500000,16119.000000,47796.000000,211.683000,4.854000,16.634000
max,77549.000000,32947.000000,46064.000000,73370.000000,913.620000,15.026000,35.692000


## 10. Save the Final Dataset

The merged dataset is saved as a CSV file for later use in forecasting and visualization.

In [22]:
merged.to_csv("germany_energy_weather_2012_2020.csv", index=False)
print("Saved file: germany_energy_weather_2012_2020.csv")

Saved file: germany_energy_weather_2012_2020.csv
